
# Interpretable Machine Learning (IML) Tutorial

This tutorial introduces Interpretable Machine Learning (IML) to help learners understand how and why models make predictions. By the end, participants will appreciate the importance of transparency, trust, and responsible AI, especially in high-stakes domains.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import shap

# Install lime if not already installed
try:
    from lime.lime_tabular import LimeTabularExplainer
except ModuleNotFoundError:
    !pip install lime
    from lime.lime_tabular import LimeTabularExplainer

plt.rcParams["figure.figsize"] = (8,6)


## Load Dataset

In [ ]:

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

In [ ]:
print(f"Dataset shape: {X.shape}")
print("Target classes: 0 = malignant, 1 = benign\n")

In [ ]:
#list first five
X.head()

In [ ]:
#view columns
X.columns

In [ ]:
#view target variables
y

## Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Model Building

### Interpretable Model

#### Decision Tree

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

print("Accuracy:", dt.score(X_test, y_test))

#### Feature Importance

In [ ]:
importances = dt.feature_importances_
indices = np.argsort(importances)[-10:]

plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), X.columns[indices])
plt.title("Top 10 Features of Decision Tress Classifier")
plt.show()

### Complex Model

####Random Forest

In [ ]:

rf= RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

print("Accuracy:", rf.score(X_test, y_test))


Feature Importance

In [ ]:

importances = rf.feature_importances_
indices = np.argsort(importances)[-10:]

plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), X.columns[indices])
plt.title("Top 10 Features of raandom forest")
plt.show()



### Exercise 1
- For both the Decision Tree and Random Forest models, which feature appears most important?
- Why do you think these features influence predictions in each model?

### Permutation Importance for Random Forest

*   List item
*   List item



In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42
)

sorted_idx = perm.importances_mean.argsort()[-10:]

plt.figure(figsize=(10, 6))
plt.boxplot(
    perm.importances[sorted_idx].T,
    vert=False,
    tick_labels=X.columns[sorted_idx]
)
plt.title("Permutation Importance")
plt.xlabel("Decrease in Performance")
plt.tight_layout()
plt.show()

### Exercise 2
- Which features consistently have high permutation importance values across multiple repetitions?
- How does this compare to the feature importance from the Random Forest model?

### SHAP Explanations for Random Forest

In [ ]:

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# Corrected: Select SHAP values for the second class (class index 1) across all samples
shap.summary_plot(shap_values[:, :, 1], X_test)



### Exercise 3
- What do the red and blue colors represent?
- Which features push predictions higher?


### LIME Explanation for Random Forest

In [ ]:

lime_explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X.columns,
    class_names=data.target_names,
    mode='classification'
)

exp = lime_explainer.explain_instance(
    X_test.iloc[0].values,
    rf.predict_proba,
    num_features=10
)

exp.show_in_notebook()



### Exercise 4
- Compare SHAP and LIME outputs.
- Do they highlight the same features?


### Simple Neural Network (MLPClassifier)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

# Ensure X_train, y_train, X_test, y_test are defined for this section
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

nn_model = MLPClassifier(random_state=42, max_iter=1000) # Increased max_iter for convergence
nn_model.fit(X_train, y_train)

print("Accuracy:", nn_model.score(X_test, y_test))

#### Permutation Importance for Neural Network

In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import numpy as np

perm_nn = permutation_importance(
    nn_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42
)

sorted_idx_nn = perm_nn.importances_mean.argsort()[-10:]

plt.figure(figsize=(10, 6))
plt.boxplot(
    perm_nn.importances[sorted_idx_nn].T,
    vert=False,
    tick_labels=X.columns[sorted_idx_nn]
)
plt.title("Permutation Importance for Neural Network")
plt.xlabel("Decrease in Performance")
plt.tight_layout()
plt.show()

#### SHAP Explanations for Neural Network

In [ ]:
import shap
import pandas as pd

# Using a sample of the training data as background for KernelExplainer (can be computationally intensive)
explainer_nn = shap.KernelExplainer(nn_model.predict_proba, shap.sample(X_train, 100))
shap_values_nn = explainer_nn.shap_values(shap.sample(X_test, 100)) # Explaining a sample of X_test

# Corrected: Select SHAP values for the second class (class index 1)
shap.summary_plot(shap_values_nn[:, :, 1], shap.sample(X_test, 100))

#### LIME Explanations for Neural Network

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
import pandas as pd

lime_explainer_nn = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X.columns,
    class_names=data.target_names,
    mode='classification'
)

# Explain a specific instance (e.g., the first instance in X_test)
exp_nn = lime_explainer_nn.explain_instance(
    X_test.iloc[0].values,
    nn_model.predict_proba,
    num_features=10
)

exp_nn.show_in_notebook()

### Exercise 5
- Compare the feature importance, permutation importance, SHAP, and LIME outputs for the Neural Network model.
- Are there any discrepancies between these methods, and if so, what might cause them?


## Quiz Questions

1. What is the difference between global and local interpretability?  
2. Which method (SHAP or LIME) provides global explanations?  
3. Why might two interpretability methods disagree?  
4. What is one limitation of feature importance?  
